# Scraper Atletas

Importar librerías

In [1]:
import httpx
import lxml.html as lh
import os
from tqdm import tqdm
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

Ingresar Credenciales

In [12]:
USER = os.getenv("USER")
PASSWORD = os.getenv("PASSWORD")
LOGIN_URL = "https://atletismo.usplat.cl/login"

In [13]:
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
# options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")

In [14]:
# 1. Ir a la página de login
driver = webdriver.Chrome(service=service, options=options)
driver.get(LOGIN_URL)
wait = WebDriverWait(driver, 10)

# 2. Esperar a que cargue el formulario de login
wait.until(EC.presence_of_element_located((By.NAME, "email")))

# 2. Ingresar credenciales
usuario = wait.until(EC.presence_of_element_located((By.NAME, "email")))
contraseña = driver.find_element(By.NAME, "password")

usuario.send_keys(USER)
contraseña.send_keys(PASSWORD)

# 3. Hacer clic en "Iniciar sesión"
boton_login = driver.find_element(By.XPATH, '//button[@type="submit"]')
boton_login.click()

# 4. Esperar a que se cargue la página principal
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "logo-usplat")))
cookies = driver.get_cookies()

# 5. Extraer numero de atletas

counter = wait.until(EC.presence_of_element_located((By.ID, "counter")))
numero_atletas_texto = counter.text
NUMERO_ATLETAS = int(numero_atletas_texto.replace(".", ""))

# 6. Crear cliente httpx y agregar cookies + headers
cookies_dict = {cookie['name']: cookie['value'] for cookie in cookies}
user_agent = driver.execute_script("return navigator.userAgent;")

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "es-ES,es;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
}

client = httpx.Client(cookies=cookies_dict, headers=headers, follow_redirects=True)

Obtener datos de un atleta

In [15]:
# 5. Hacer una solicitud a la página de atleta

response = client.get("https://atletismo.usplat.cl/atleta/8049")
tree = lh.fromstring(response.content)
nombre = tree.xpath('//div[@class="nombre"]/text()')
apellido = tree.xpath('//div[@class="apellido"]/text()')
fecha_de_nacimiento = tree.xpath('//div[@class="label" and text()="Edad"]/following-sibling::div/p[2]/text()')
pais = tree.xpath('//div[@class="label" and text()="Pais"]/following-sibling::div/text()')
colegio = tree.xpath('//div[@class="label" and text()="Colegio"]/following-sibling::div/text()')
club = tree.xpath('//div[@class="label" and text()="Club"]/following-sibling::div/text()')
club_master = tree.xpath('//div[@class="label" and text()="Club Master"]/following-sibling::div/text()')

print("Nombre:", nombre[0] if nombre else '')
print("Apellido:", apellido[0] if apellido else '')
print("Fecha de nacimiento:", fecha_de_nacimiento[0] if fecha_de_nacimiento else '')
print("Pais:", pais[0] if pais else '')
print("Colegio:", colegio[0] if colegio else '')
print("Club:", club[0] if club else '')
print("Club Master:", club_master[0] if club_master else '')

      

Nombre: Leslie Rigoberto
Apellido: Encina Quintana
Fecha de nacimiento: (02 Ene 1982)
Pais: Chile
Colegio: 
Club: Ejercito de Chile
Club Master: Círculo Atlético Royal


In [16]:
NUMERO_ATLETAS

81620

In [17]:
atletas_antiguos = pd.read_csv("../BD/Tablas/atletas.csv")
FIRST_ID = atletas_antiguos["id_atleta"].max() + 1
# NUMERO_ATLETAS = 81561
LAST_ID = NUMERO_ATLETAS + 196
ID_RANGE = range(FIRST_ID, LAST_ID + 1)

scraped_ids = atletas_antiguos["id_atleta"].astype(int)

ids_por_scrapear = (
    pd.Index(range(FIRST_ID, LAST_ID + 1))
    .difference(scraped_ids)
    .tolist()
)
len(ids_por_scrapear)


59

## Scraper Síncrono

In [18]:
data = []

for i in tqdm(ids_por_scrapear):
    response = client.get(f"https://atletismo.usplat.cl/atleta/{i}")
    tree = lh.fromstring(response.content)
    
    nombre = tree.xpath('//div[@class="nombre"]/text()')
    apellido = tree.xpath('//div[@class="apellido"]/text()')
    fecha_de_nacimiento = tree.xpath('//div[@class="label" and text()="Edad"]/following-sibling::div/p[2]/text()')
    pais = tree.xpath('//div[@class="label" and text()="Pais"]/following-sibling::div/text()')
    colegio = tree.xpath('//div[@class="label" and text()="Colegio"]/following-sibling::div/text()')
    club = tree.xpath('//div[@class="label" and text()="Club"]/following-sibling::div/text()')
    club_master = tree.xpath('//div[@class="label" and text()="Club Master"]/following-sibling::div/text()')

    if nombre:
        data.append({
            "id_atleta": i,
            "Nombre": nombre[0] if nombre else '',
            "Apellido": apellido[0] if apellido else '',
            "Fecha de Nacimiento": fecha_de_nacimiento[0] if fecha_de_nacimiento else '',
            "Pais": pais[0] if pais else '',
            "Colegio": colegio[0] if colegio else '',
            "Club": club[0] if club else '',
            "Club Master": club_master[0] if club_master else ''
        })
    else:
        print(f"Atleta con ID {i} no encontrado o sin datos.")

100%|██████████| 59/59 [00:05<00:00,  9.97it/s]


## Scraper Asíncrono

In [19]:
# CONCURRENT_REQUESTS = 50

# semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)
# data = []
# errors = dict()
# not_found = []

# async def fetch_atleta(client, i):
#     url = f"https://atletismo.usplat.cl/atleta/{i}"

#     async with semaphore:
#         try:
#             resp = await client.get(url, timeout=100)
#             if resp.status_code != 200:
#                 if errors.get(resp.status_code):
#                     errors[resp.status_code].append(i)
#                 else:
#                     errors[resp.status_code] = [i]
#                 return 
#             tree = lh.fromstring(resp.content)
#             nombre = tree.xpath('//div[@class="nombre"]/text()')
#             if not nombre:
#                 not_found.append(i)
#                 return  # página no válida

#             apellido = tree.xpath('//div[@class="apellido"]/text()')
#             fecha_de_nacimiento = tree.xpath('//div[@class="label" and text()="Edad"]/following-sibling::div/p[2]/text()')
#             pais = tree.xpath('//div[@class="label" and text()="Pais"]/following-sibling::div/text()')
#             colegio = tree.xpath('//div[@class="label" and text()="Colegio"]/following-sibling::div/text()')
#             club = tree.xpath('//div[@class="label" and text()="Club"]/following-sibling::div/text()')
#             club_master = tree.xpath('//div[@class="label" and text()="Club Master"]/following-sibling::div/text()')

#             return {
#                 "id_atleta": i,
#                 "Nombre": nombre[0].strip(),
#                 "Apellido": apellido[0].strip() if apellido else '',
#                 "Fecha de Nacimiento": fecha_de_nacimiento[0].strip() if fecha_de_nacimiento else '',
#                 "Pais": pais[0].strip() if pais else '',
#                 "Colegio": colegio[0].strip() if colegio else '',
#                 "Club": club[0].strip() if club else '',
#                 "Club Master": club_master[0].strip() if club_master else ''
#             }

#         except Exception as e:
#             print(f"Error en ID {i}: {e}")
#             return

# async def main(ID_RANGE):
#     async with httpx.AsyncClient(http2=False, cookies=cookies_dict, headers=headers, follow_redirects=True) as client:
#         tasks = [fetch_atleta(client, i) for i in ID_RANGE]
#         for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
#             result = await coro
#             if result:
#                 data.append(result)
#     # Guardar resultados
#     atletas_nuevos = pd.DataFrame(data)
#     return atletas_nuevos

In [20]:
atletas_nuevos = pd.DataFrame(data)
atletas_nuevos

,id_atleta,Nombre,Apellido,Fecha de Nacimiento,Pais,Colegio,Club,Club Master
0,81758,Mai,Chalhub Dahma,(25 Abr 2011),Chile,Lycée Jean Mermoz,,
1,81759,Luis Emilio,Neira Andrade,(21 Abr 2013),Chile,Escuela Básica Pueblo Seco,,
2,81760,Alonso Elias,Gomez Cuevas,(13 Oct 2013),Chile,Escuela Básica Pueblo Seco,,
3,81761,Francisco Javier,Baeza Valenzuela,(15 Mar 2012),Chile,Escuela Básica Pueblo Seco,,
4,81762,Ignacio -,Villagran Diaz,(10 Jun 2012),Chile,Escuela Zemita,,
5,81763,Trinidad -,Pacheco Flores,(22 Ago 2015),Chile,Escuela Básica San Ignacio de Palomares,,
6,81764,Matilde Hope,Sanhueza Flores,(02 Feb 2013),Chile,Liceo Bicentenario Polivalente San Nicolás,,
7,81765,Jean Franco Alonso,Vergara Araneda,(06 Ago 2012),Chile,Escuela Básica El Casino,,
8,81766,Christofer Alexander,Lagos Burgos,(16 Abr 2013),Chile,Escuela Básica El Casino,,
9,81767,Monserrat Alejandra,Monsalve Mardones,(20 Jun 2014),Chile,Escuela F-393 Navidad,,


In [21]:
Traduccion = {"Ene":"Jan", "Abr":"Apr","Ago":"Aug","Dic":"Dec"}
atletas_nuevos = atletas_nuevos.sort_values(by="id_atleta")
atletas_nuevos["Fecha de Nacimiento"] = atletas_nuevos["Fecha de Nacimiento"].str.strip("()").str.strip()
atletas_nuevos["Fecha de Nacimiento"] = pd.to_datetime(atletas_nuevos["Fecha de Nacimiento"].replace(Traduccion, regex=True), format="%d %b %Y", errors='coerce')
atletas_nuevos

,id_atleta,Nombre,Apellido,Fecha de Nacimiento,Pais,Colegio,Club,Club Master
0,81758,Mai,Chalhub Dahma,2011-04-25,Chile,Lycée Jean Mermoz,,
1,81759,Luis Emilio,Neira Andrade,2013-04-21,Chile,Escuela Básica Pueblo Seco,,
2,81760,Alonso Elias,Gomez Cuevas,2013-10-13,Chile,Escuela Básica Pueblo Seco,,
3,81761,Francisco Javier,Baeza Valenzuela,2012-03-15,Chile,Escuela Básica Pueblo Seco,,
4,81762,Ignacio -,Villagran Diaz,2012-06-10,Chile,Escuela Zemita,,
5,81763,Trinidad -,Pacheco Flores,2015-08-22,Chile,Escuela Básica San Ignacio de Palomares,,
6,81764,Matilde Hope,Sanhueza Flores,2013-02-02,Chile,Liceo Bicentenario Polivalente San Nicolás,,
7,81765,Jean Franco Alonso,Vergara Araneda,2012-08-06,Chile,Escuela Básica El Casino,,
8,81766,Christofer Alexander,Lagos Burgos,2013-04-16,Chile,Escuela Básica El Casino,,
9,81767,Monserrat Alejandra,Monsalve Mardones,2014-06-20,Chile,Escuela F-393 Navidad,,


In [22]:
all_athletes = pd.concat([atletas_antiguos, atletas_nuevos], ignore_index=True)
all_athletes = all_athletes.drop_duplicates(subset='id_atleta', keep='last')
all_athletes = all_athletes.sort_values(by='id_atleta')
all_athletes.to_csv('../BD/Tablas/atletas.csv', index=False)
